# Activity 2: Tokens and Embeddings

**Week 6 Day 3 · The two things text becomes before a model can touch it**

In Activity 1 you printed this and moved on:

```
prompt tokens: 44
completion tokens: 31
```

That is the billing unit of this entire industry, and nobody has told you what it is yet. This notebook fixes that, and then goes one step further into the other representation you will meet constantly: **embeddings**.

Both answer the same question, "how do you turn text into numbers," but for completely different purposes:

- **Tokens** are how text gets *fed to* a model. They are chunks of characters with ID numbers. They decide what you pay and what fits.
- **Embeddings** are how text gets *compared*. They are lists of numbers that capture meaning, so that two sentences saying the same thing in different words land near each other.

## What you will learn

- What a token actually is, and how to count them before you send a request
- Why models are strangely bad at spelling, and why Spanish costs more than English
- What an embedding is, and why two sentences with no shared words can score as nearly identical
- How to generate embeddings with an API and with a small model on your own machine
- Where each one is the right tool, and where neither is

---
## Setup

`tiktoken` and `model2vec` were installed in [Activity 0](./Activity_0_Environment_and_API_Setup.md). The OpenAI parts reuse your `OPENAI_API_KEY`.

In [ ]:
import os

import numpy as np
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("ready")

---
# Part 1: Tokens

A model does not see letters and it does not see words. Before anything else happens, your text is chopped into **tokens**, chunks of characters that the model has an ID number for. The model only ever works with those IDs.

`tiktoken` is OpenAI's tokenizer library, and it lets you run that exact chopping step yourself, locally, with no API call and no cost.

In [ ]:
encoder = tiktoken.encoding_for_model("gpt-4o-mini")

token_ids = encoder.encode("The insured filed a claim.")
print("token ids:", token_ids)
print("how many:", len(token_ids))

Those integers are literally what gets sent. Now decode them one at a time to see where the cuts landed.

In [ ]:
for tid in token_ids:
    print(f"{tid:>7}  {encoder.decode([tid])!r}")

Notice the leading spaces. `' insured'` is one token *including* the space in front of it, which is why token counts do not match word counts even when the split looks word-shaped.

That sentence was made of common words, so the split was clean. Try words that are longer or rarer.

In [ ]:
for word in ["cat", "Hartford", "deductible", "subrogation", "CLM_101"]:
    ids = encoder.encode(word)
    pieces = [encoder.decode([i]) for i in ids]
    print(f"{word:<14} {len(ids)} token(s)  {pieces}")

This is **BPE** (Byte Pair Encoding) doing its job. The tokenizer was built by finding the most frequent character sequences in a huge pile of text and giving each one an ID. Common words like `cat` earned their own token. `deductible` did not, so it gets rebuilt from `ded` + `uct` + `ible`. `CLM_101` is not a word at all, so it shatters into four pieces.

The practical rule: **common text is cheap, unusual text is expensive.** Identifiers, code, medical terms, and rare names all cost more tokens than their length suggests.

This also explains a famous failure. Ask a model how many times the letter "r" appears in "strawberry" and it often gets it wrong. Look at why.

In [ ]:
print([encoder.decode([i]) for i in encoder.encode("strawberry")])

The model never receives the letters `s-t-r-a-w-b-e-r-r-y`. It receives three chunks. Asking it to count characters is like asking someone to count the letters in a word they only ever heard spoken. It is not a reasoning failure, it is a representation failure, and knowing the difference tells you which problems to hand a model and which to solve with three lines of Python.

---
## Counting tokens before you spend money

You can count tokens locally, before sending anything. This is how you estimate cost and check that a prompt will fit.

In [ ]:
CLAIM_NOTE = (
    "Insured reports rear-end collision at low speed in a parking lot. "
    "Bumper cover cracked, no airbag deployment, other party's insurance "
    "already confirmed liability. Insured requests expedited repair "
    "authorization due to upcoming work travel."
)

print("characters:", len(CLAIM_NOTE))
print("words:     ", len(CLAIM_NOTE.split()))
print("tokens:    ", len(encoder.encode(CLAIM_NOTE)))

33 words becomes 44 tokens. The usual rough estimate is that one token is about three quarters of a word in English, and this note is close to that.

Go back to Activity 1 and look at what `completion.usage.prompt_tokens` reported for this same claim note. That number is not quite 44, because the system message and some per-message formatting are counted too. But you can now predict your bill to within a few percent without making a single API call.

---
## Different models, different tokenizers

A tokenizer is a fixed piece of a model, chosen when it was trained. Change models and the same text costs a different number of tokens.

In [ ]:
newer = tiktoken.get_encoding("o200k_base")   # used by gpt-4o and gpt-4o-mini
older = tiktoken.get_encoding("cl100k_base")  # used by the gpt-4 and gpt-3.5 generation

print("o200k_base :", len(newer.encode(CLAIM_NOTE)), "tokens, vocabulary", newer.n_vocab)
print("cl100k_base:", len(older.encode(CLAIM_NOTE)), "tokens, vocabulary", older.n_vocab)

A larger vocabulary means more sequences got their own token, so the same text needs fewer of them. The difference is small on plain English prose and gets much larger on code and non-English text.

Speaking of which. Here is the same sentence in two languages.

In [ ]:
english = "The insured's basement flooded after a pipe burst overnight."
spanish = "El sótano del asegurado se inundó después de que una tubería se rompiera durante la noche."

print("English:", len(encoder.encode(english)), "tokens")
print("Spanish:", len(encoder.encode(spanish)), "tokens")

The same meaning costs roughly twice as much in Spanish. The tokenizer's vocabulary was built mostly from English text, so English gets efficient whole-word tokens while other languages get chopped into fragments.

That is a real budget line for a company serving policyholders in more than one language, and it is invisible unless you know to look for it.

### Checkpoint: why tokens matter to you

| Because | Concretely |
| :--- | :--- |
| You pay per token | Both directions, on every call, including the resent history |
| The context window is measured in tokens | System prompt, history, retrieved documents, and tool results all compete for one budget |
| Not all text costs the same | Code, IDs, rare words, and non-English text cost more than their length suggests |
| Models cannot see letters | Character-level tasks are the wrong job for a model |

---
# Part 2: Embeddings

Tokens get text *into* a model. They are useless for comparing meaning, because token IDs are just labels. ID 1234 is not more similar to ID 1235 than to ID 9999, any more than house number 12 is similar to house number 13.

An **embedding** is different. It is a list of floating point numbers, produced by a model trained so that text with similar meaning produces similar numbers. That single property is what makes semantic search, clustering, deduplication, and recommendations possible.

Start with one sentence.

In [ ]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="The insured's basement flooded after a pipe burst.",
)
vector = response.data[0].embedding

print("dimensions:", len(vector))
print("first 8 values:", [round(v, 4) for v in vector[:8]])

1,536 numbers standing in for one sentence. No individual number means anything you could name, there is no "flooding" dimension. The meaning lives in the whole pattern, in where that point sits relative to every other point.

Which means a single embedding on its own is useless. Embeddings only do work in *comparison*.

## Comparing two embeddings

The standard measure is **cosine similarity**, which asks how closely two vectors point in the same direction, ignoring how long they are. It runs from 1.0 (identical direction) through 0.0 (unrelated) to -1.0 (opposite).

In [ ]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
SENTENCES = [
    "The insured's basement flooded after a pipe burst.",
    "Water damage throughout the lower level of the home following a plumbing failure.",
    "The rear bumper was cracked in a parking lot collision.",
]

response = client.embeddings.create(model="text-embedding-3-small", input=SENTENCES)
vectors = [item.embedding for item in response.data]

print(f"flood vs water damage : {cosine_similarity(vectors[0], vectors[1]):.3f}")
print(f"flood vs bumper       : {cosine_similarity(vectors[0], vectors[2]):.3f}")

The first pair should score high and the second should be much lower.

Now look at why that is remarkable. Compare the actual words in the first pair.

In [ ]:
def word_overlap(a, b):
    words_a = set(a.lower().replace(".", "").split())
    words_b = set(b.lower().replace(".", "").split())
    return words_a & words_b


print("shared words:", word_overlap(SENTENCES[0], SENTENCES[1]))

The only words those two sentences share are `the` and `a`.

Not one meaningful word in common. No `flood`, no `basement`, no `water`, nothing a keyword index could catch. One says "basement flooded after a pipe burst," the other says "water damage on the lower level following a plumbing failure." They describe the same event, and the embeddings placed them close together anyway.

That is the whole value proposition. Any technique built on matching words, `LIKE '%flood%'`, keyword search, a `WHERE` clause, would score these two as unrelated. Embeddings score them as nearly the same thing, because they are.

Hold onto that. Tomorrow's lab is built entirely on this one property.

---
## An embedding model on your own machine

The OpenAI call above sent your text over the network and billed you. That is not the only option. Activity 3 asks the same question about chat models, and it is worth seeing the answer here first, because for embeddings the local option is far less of a compromise.

Embedding models are *small*. The one below is a few tens of megabytes and runs on CPU, no GPU and no PyTorch required, which matters on a modest VM.

In [ ]:
from model2vec import StaticModel

embedder = StaticModel.from_pretrained("minishlab/potion-base-8M")
local_vectors = embedder.encode(SENTENCES)

print("dimensions:", local_vectors.shape[1])
print(f"flood vs water damage : {cosine_similarity(local_vectors[0], local_vectors[1]):.3f}")
print(f"flood vs bumper       : {cosine_similarity(local_vectors[0], local_vectors[2]):.3f}")

Same conclusion, different scale. You should see the related pair land somewhere around 0.4 to 0.5 and the unrelated pair below 0.1, so the *ranking* matches the API exactly even though the raw numbers do not.

That last point is important and often confuses people: **cosine scores are not comparable across models.** A 0.55 from this model and a 0.55 from OpenAI mean different things. What transfers is the ordering, and ordering is all most applications need.

Two honest caveats about this particular model. It produces 256 dimensions rather than 1,536, which makes it cheaper to store and faster to compare, but coarser. And it is a *static* model: each word contributes the same vector regardless of context, so it cannot tell the "bank" of a river from a bank that holds money. The heavier open alternative, `all-MiniLM-L6-v2` run through `sentence-transformers`, is context-aware and better, but it needs PyTorch and roughly two gigabytes of dependencies, which is a poor trade on a classroom VM for a lesson this model teaches perfectly well.

## Choosing between them

| | OpenAI `text-embedding-3-small` | Local `potion-base-8M` |
| :--- | :--- | :--- |
| Dimensions | 1,536 | 256 |
| Where it runs | OpenAI's servers | Your machine, CPU |
| Cost | Per token | Free after download |
| Data leaves your network | Yes | No |
| Quality | Stronger, context-aware | Coarser, static |
| Storage per million records | Larger vectors, larger index | Roughly six times smaller |

The decision looks exactly like the one from Activity 3, with one difference: the quality gap between a frontier chat model and a small local one is enormous, while the gap between embedding models is much narrower. Local embeddings are a genuinely competitive choice far more often than local chat models are.

---
## When embeddings are the right tool

Embeddings are for **comparison at scale**. If your problem can be phrased as "find things that are alike," they probably fit.

| Use case | What you compare |
| :--- | :--- |
| Semantic search | A question against a corpus of documents |
| RAG | The same, then the winners get fed to an LLM (tomorrow) |
| Deduplication | Every record against every other record |
| Clustering | Grouping claim notes by theme with no labels |
| Classification | New text against labelled examples |
| Recommendation | One item against a catalogue |

And where they are the wrong tool, which matters just as much:

- **Exact matching.** Finding claim `CLM_101` is a database lookup. Do not embed an ID.
- **Structured filters.** "Claims over $10,000 in Connecticut" is a `WHERE` clause. Embeddings would make it slower, more expensive, and wrong.
- **Anything generative.** Embeddings compare text, they never produce it. Writing a summary needs an LLM.
- **Explaining a decision.** A cosine score of 0.61 is not a reason a human can act on.

The pattern to notice: embeddings answer "which of these is most like that," and nothing else. Every real system pairs them with something that does the rest.

---
# Your Turn

Work in your own copy under `student-work/week6/day3/`.

1. Write a function `count_tokens(text)` that returns the token count for `gpt-4o-mini`. Run it on three prompts of your own and note how the count compares to the word count. Then find a piece of text where the ratio is unusually high, and explain why it is.
2. Add two more sentences to `SENTENCES`: one that clearly means the same as the bumper claim in different words, and one about something unrelated (weather, lunch, anything). Rebuild the embeddings and print all pairwise similarities. Does the model group them the way you would?

**Stretch goal:** embed the *same* sentence twice in two separate API calls and compare the two vectors. Are they identical? Now do the same with a chat completion, sending the same prompt twice. Which of the two is deterministic, and why does that difference matter when you are deciding what to cache?

## What you did

- Ran BPE tokenization yourself and saw where the cuts land.
- Explained model failures on spelling and the higher cost of non-English text from first principles.
- Counted tokens locally to estimate cost and context usage before sending a request.
- Generated embeddings and compared them with cosine similarity.
- Watched two sentences with almost no shared words score as nearly identical.
- Ran an embedding model locally with no API, no GPU, and no network call.
- Mapped out where embeddings are the right tool and where they are the wrong one.

**Next:** [Activity 3](./Activity_3_Local_Models_with_HuggingFace.ipynb) takes the local-versus-API question from embeddings to full chat models, by loading a model's weights and running it yourself.